# Case Study 3 — Cement Clinker Decarbonization Scenarios

Companion: **study.md**. Industrial-system LCA with three scenarios (baseline,
alternative fuel, CCS), decomposing GWP into calcination / combustion /
electricity to expose the process-chemistry floor. Uses Brightway to build and
characterize each scenario as a separate foreground.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import bw2data as bd
import bw2calc as bc
import bw2io as bi

13:20:25-0400

 [

warning  

] 

Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.

In [2]:
PROJECT = "cs3-cement"
if PROJECT not in bd.projects:
    if "bw25-tutorials" in bd.projects:
        bd.projects.set_current("bw25-tutorials")
        bd.projects.copy_project(PROJECT, switch=True)
    else:
        bi.remote.install_project("ecoinvent-3.10-biosphere", PROJECT)
        bd.projects.set_current(PROJECT)
else:
    bd.projects.set_current(PROJECT)
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)
co2 = next(f for f in bio if f["name"] == "Carbon dioxide, fossil" and f["categories"] == ("air",))
print("project:", bd.projects.current)

project:

cs3-cement

## Scenario parameters

In [3]:
scen = pd.read_csv(Path.cwd() / "data" / "cement_scenarios.csv").set_index("parameter")
print(scen.to_string())
SCEN_COLS = ["baseline", "alt_fuel", "ccs"]

                 baseline  alt_fuel    ccs                    unit                                               notes
parameter                                                                                                             
clinker_ratio       0.900     0.900  0.900  kg clinker / kg cement         rest is supplementary cementitious material
calcination_co2     0.530     0.530  0.530     kg CO2 / kg clinker  process CO2 from limestone (unavoidable chemistry)
fuel_energy         3.400     3.400  3.600         MJ / kg clinker       kiln thermal demand (CCS adds parasitic load)
coal_share          1.000     0.400  0.400                fraction                      share of kiln energy from coal
coal_ci             0.096     0.096  0.096             kg CO2 / MJ                              coal combustion factor
altfuel_ci          0.020     0.020  0.020             kg CO2 / MJ                waste-derived fuel (partly biogenic)
elec_use            0.110     0.110  0.280      

## Decompose GWP per scenario (calcination / combustion / electricity)

In [4]:
def decompose(col):
    p = scen[col]
    calc = p["clinker_ratio"] * p["calcination_co2"]
    comb = (p["clinker_ratio"] * p["fuel_energy"] *
            (p["coal_share"] * p["coal_ci"] + (1 - p["coal_share"]) * p["altfuel_ci"]))
    elec = p["elec_use"] * p["grid_ci"]
    captured = p["ccs_capture"] * (calc + comb)
    # CCS captures a fraction of calcination+combustion; electricity is the penalty
    calc_net = calc * (1 - p["ccs_capture"])
    comb_net = comb * (1 - p["ccs_capture"])
    return {"calcination": calc_net, "combustion": comb_net, "electricity": elec}

decomp = pd.DataFrame({c: decompose(c) for c in SCEN_COLS}).T
decomp["total"] = decomp.sum(axis=1)
print(decomp.round(4).to_string())

          calcination  combustion  electricity   total
baseline       0.4770      0.2938       0.0462  0.8170
alt_fuel       0.4770      0.1542       0.0462  0.6774
ccs            0.0716      0.0245       0.0560  0.1520

## Build each scenario in Brightway and cross-check the totals

In [5]:
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))

def build_and_score(col):
    d = decompose(col)
    total = sum(d.values())
    DB = f"cs3_{col}"
    if DB in bd.databases:
        del bd.databases[DB]
    bd.Database(DB).write({
        (DB, "cement"): {"name": f"cement ({col})", "unit": "kilogram", "exchanges": [
            {"input": (DB, "cement"), "amount": 1.0, "type": "production"},
            {"input": co2.key, "amount": total, "type": "biosphere"}]},
    })
    act = bd.get_node(database=DB, code="cement")
    l = bc.LCA({act: 1}, method=gwp); l.lci(); l.lcia()
    return l.score

bw_scores = {c: build_and_score(c) for c in SCEN_COLS}
for c in SCEN_COLS:
    assert abs(bw_scores[c] - decomp.loc[c, "total"]) / decomp.loc[c, "total"] < 1e-4
print("Brightway totals match hand decomposition:")
for c in SCEN_COLS:
    print(f"   {c:10s} {bw_scores[c]:.4f} kg CO2-eq/kg cement")

13:20:26-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

Vacuuming database            

Brightway totals match hand decomposition:

   baseline   0.8170 kg CO2-eq/kg cement

   alt_fuel   0.6774 kg CO2-eq/kg cement

   ccs        0.1520 kg CO2-eq/kg cement

## Stacked bar: where each scenario's emissions come from

In [6]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
bottom = np.zeros(len(SCEN_COLS))
colors = {"calcination": "#C44E52", "combustion": "#DD8452", "electricity": "#4C72B0"}
for part in ["calcination", "combustion", "electricity"]:
    vals = decomp[part].values
    ax.bar(SCEN_COLS, vals, bottom=bottom, label=part, color=colors[part])
    bottom += vals
# annotate the irreducible calcination floor of the baseline
floor = decompose("baseline")["calcination"]
ax.axhline(floor, color="k", ls=":", lw=1)
ax.text(2.4, floor*1.02, "baseline calcination", fontsize=8, ha="right")
ax.set_ylabel("kg CO2-eq / kg cement")
ax.set_title("Cement GWP by source across decarbonization scenarios")
ax.legend()
plt.tight_layout(); plt.savefig("cs3_scenarios.png", dpi=130, bbox_inches="tight")
print("saved cs3_scenarios.png"); plt.show()

saved cs3_scenarios.png

C:\Users\derne\AppData\Local\Temp\ipykernel_18784\1139121995.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved cs3_scenarios.png"); plt.show()


## Reduction vs baseline

In [7]:
base_total = decomp.loc["baseline", "total"]
red = pd.DataFrame({
    "scenario": SCEN_COLS,
    "total": [decomp.loc[c, "total"] for c in SCEN_COLS],
    "reduction_vs_baseline": [f"{(1 - decomp.loc[c,'total']/base_total):.0%}" for c in SCEN_COLS],
})
print(red.round(4).to_string(index=False))

scenario  total reduction_vs_baseline
baseline 0.8170                    0%
alt_fuel 0.6774                   17%
     ccs 0.1520                   81%

## Sensitivity: clinker ratio, capture rate, grid intensity (on the CCS scenario)

In [8]:
def ccs_total(clinker_ratio=None, ccs_capture=None, grid_ci=None):
    p = scen["ccs"].copy()
    if clinker_ratio is not None: p["clinker_ratio"] = clinker_ratio
    if ccs_capture is not None: p["ccs_capture"] = ccs_capture
    if grid_ci is not None: p["grid_ci"] = grid_ci
    calc = p["clinker_ratio"] * p["calcination_co2"] * (1 - p["ccs_capture"])
    comb = (p["clinker_ratio"] * p["fuel_energy"] *
            (p["coal_share"]*p["coal_ci"] + (1-p["coal_share"])*p["altfuel_ci"])) * (1 - p["ccs_capture"])
    elec = p["elec_use"] * p["grid_ci"]
    return calc + comb + elec

base = ccs_total()
tor = []
for name, kw_lo, kw_hi in [
    ("clinker_ratio", {"clinker_ratio": 0.70}, {"clinker_ratio": 0.95}),
    ("ccs_capture", {"ccs_capture": 0.70}, {"ccs_capture": 0.95}),
    ("grid_ci", {"grid_ci": 0.10}, {"grid_ci": 0.42}),
]:
    lo, hi = ccs_total(**kw_lo) - base, ccs_total(**kw_hi) - base
    tor.append({"param": name, "low": lo, "high": hi, "swing": abs(hi-lo)})
tdf = pd.DataFrame(tor).sort_values("swing")
print(tdf.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(tdf))
ax.barh(y, tdf["high"], color="#C44E52")
ax.barh(y, tdf["low"], color="#4C72B0")
ax.set_yticks(y); ax.set_yticklabels(tdf["param"]); ax.axvline(0, color="k", lw=1)
ax.set_xlabel("Δ kg CO2-eq/kg vs CCS baseline"); ax.set_title("CCS-scenario sensitivity")
plt.tight_layout(); plt.savefig("cs3_tornado.png", dpi=130, bbox_inches="tight")
print("saved cs3_tornado.png"); plt.show()

        param     low    high  swing
clinker_ratio -0.0213  0.0053 0.0267
      grid_ci -0.0280  0.0616 0.0896
  ccs_capture  0.0960 -0.0640 0.1601

saved cs3_tornado.png

C:\Users\derne\AppData\Local\Temp\ipykernel_18784\3955371948.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved cs3_tornado.png"); plt.show()


## Conclusion + export

In [9]:
decomp.round(4).to_csv("cs3_summary.csv")
print(decomp.round(4).to_string())
print("\nCalcination CO2 is the hard floor: alt-fuel only cuts combustion; CCS is the")
print("only lever that reaches process CO2, at an electricity penalty. Clinker")
print("substitution (lower clinker_ratio) and capture rate dominate the sensitivity.")

          calcination  combustion  electricity   total
baseline       0.4770      0.2938       0.0462  0.8170
alt_fuel       0.4770      0.1542       0.0462  0.6774
ccs            0.0716      0.0245       0.0560  0.1520


Calcination CO2 is the hard floor: alt-fuel only cuts combustion; CCS is the

only lever that reaches process CO2, at an electricity penalty. Clinker

substitution (lower clinker_ratio) and capture rate dominate the sensitivity.